<a href="https://colab.research.google.com/github/Bukunmi2108/ml-journey/blob/main/research/p2/qlora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

QLoRa

In [1]:
!pip install -U bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [2]:
import torch
from peft import LoraConfig
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [3]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
dataset = load_dataset("tatsu-lab/alpaca", split="train")

README.md:   0%|          | 0.00/7.47k [00:00<?, ?B/s]

data/train-00000-of-00001-a09b74b3ef9c3b(…):   0%|          | 0.00/24.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

In [4]:
bnb_config = BitsAndBytesConfig(load_in_4bit=True,
                                bnb_4bit_compute_dtype=torch.float16,
                                bnb_4bit_quant_type="nf4",
                                bnb_4bit_use_double_quant=True)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [6]:
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:36: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
model.config.use_cache = False

In [8]:
def format_example(ex):
  instruction = ex["instruction"]
  input_text = ex["input"]
  output = ex["output"]

  prompt = instruction if not input_text else f"{instruction}\n\nInput:\n{input_text}"
  return {"text": f"<|user|>\n{prompt}\n<|assistant|>\n{output}{tokenizer.eos_token}"}

In [9]:
dataset = dataset.map(format_example, remove_columns=dataset.column_names)
dataset = dataset.train_test_split(test_size=0.20, seed=42)

Map:   0%|          | 0/52002 [00:00<?, ? examples/s]

HyperParameters setup

In [10]:
lora_alpha = 32
lora_rank = 16
eval_steps = 100
max_length = 512
lora_dropout = 0.05
learning_rate = 2e-4
num_train_epochs = 3
model_output_dir = "model/qlora_qwen"
lora_target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

In [11]:
lora_config = LoraConfig(r=lora_rank, lora_alpha=lora_alpha, lora_dropout=lora_dropout, bias="none", task_type="CAUSAL_LM", target_modules=lora_target_modules)

In [12]:
training_args = SFTConfig(output_dir=model_output_dir,
                          dataset_text_field="text",
                          max_length=max_length,
                          per_device_train_batch_size=1,
                          gradient_accumulation_steps=8,
                          num_train_epochs=num_train_epochs,
                          learning_rate=learning_rate,
                          logging_steps=10,
                          save_steps=100,
                          eval_strategy="steps",
                          eval_steps=eval_steps,
                          fp16=True,
                          optim="paged_adamw_8bit",
                          report_to="none")

In [13]:
trainer = SFTTrainer(model=model, args=training_args, train_dataset=dataset["train"], eval_dataset=dataset["test"], peft_config=lora_config)

Adding EOS to train dataset:   0%|          | 0/41601 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/41601 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/41601 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/10401 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/10401 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/10401 [00:00<?, ? examples/s]

In [14]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:80: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cpu/ops.py:132: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(bl

KeyboardInterrupt: 

In [ ]:
trainer.model.print_trainable_parameters()
trainer.save_model(model_output_dir)
tokenizer.save_pretrained(model_output_dir)